# ChronoGPT-Instruct replication — verification notebook

Run this on the GPU box (from the repo root, with the package installed:
`pip install -e '.[viz,eval]'`). It checks every implementation choice recorded
in `docs/implementation-notes.md` *before* we commit to a full training run.

Each section prints something you can sanity-check and report back. Sections are
ordered cheap → expensive (dataset/tokenizer first, model loads later).

### 0. Config — edit these if needed

In [ ]:
import os
# Run from the repo root so configs/ and tests/ paths resolve even when the
# notebook is launched from notebooks/.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
REPO_BASE = "manelalab/chrono-gpt-v1-20201231"          # a base vintage
REPO_INSTRUCT = "manelalab/chrono-gpt-instruct-v1-19991231"  # an instruct vintage
DATASET = "manelalab/ChronoInstruct-SFT"
BLOCK_SIZE = 1792
print("cwd:", os.getcwd())  # should be the repo root

### 1. Environment
Confirm stable torch + the GPU (we expect `2.x+cu126`, CUDA True, an 80GB card).

In [ ]:
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("device:", p.name, "| VRAM GB:", round(p.total_memory / 1e9, 1))

### 2. Tokenizer & the "no [pad] token" question
GPT-2 / tiktoken has **no pad token** — only `<|endoftext|>` (50256). The model's
`vocab_size` (50304) is *vocabulary padding* for GPU efficiency, NOT a pad token.
This is why we pack instead of pad (the model's forward has no attention mask).

In [ ]:
import tiktoken, json
from huggingface_hub import hf_hub_download

enc = tiktoken.get_encoding("gpt2")
print("n_vocab:", enc.n_vocab)
print("eot_token (<|endoftext|>):", enc.eot_token)
print("special tokens:", enc._special_tokens, " <- note: no [PAD]")

cfg_json = json.load(open(hf_hub_download(REPO_BASE, "config.json")))
print("\nmodel config:", cfg_json)
print("\nvocab_size", cfg_json.get("vocab_size"), "vs tokenizer n_vocab", enc.n_vocab,
      "-> the extra rows are unused vocab padding, not a pad token")

### 3. Dataset columns & sample-row shapes
Verify the real types of `conversation` (dict vs JSON string) and `label`
(this is what `data.py` must handle).

In [ ]:
import pprint
from datasets import load_dataset

ds = load_dataset(DATASET, split="train")
print("columns:", ds.column_names, "| rows:", len(ds))
row = ds[0]
for k in ds.column_names:
    v = row[k]
    print(f"\n--- {k}  (type: {type(v).__name__}) ---")
    pprint.pprint(v[:400] if isinstance(v, str) else v)

### 4. Temporal screen vs paper Table 1
After `keep_row` (label 0, confidence >=10) the total should be ~425,119 with
per-source counts near scratch 1,097 / self-instruct 67,136 / tulu 356,886.
If the total is far off, the `label` predicate needs adjusting to its real shape.

In [ ]:
from chrono_instruct.data import keep_row, source_counts

print("RAW source counts:")
for s, n in sorted(source_counts(ds).items(), key=lambda x: -x[1]):
    print(f"  {n:>9,}  {s}")

fc = source_counts(ds, after_filter=True)
print("\nAFTER screen (label==0 & confidence>=10):")
for s, n in sorted(fc.items(), key=lambda x: -x[1]):
    print(f"  {n:>9,}  {s}")
print(f"\n  total after screen: {sum(fc.values()):,}   (paper Table 1: 425,119)")

### 4b. Why Tulu under-screens — label parsing + confidence breakdown
The `label` field is stored inconsistently: scratch/self-instruct use valid JSON,
but Tulu uses single-quoted Python-dict reprs that `json.loads` rejects (so they
were silently dropped — the cause of the Tulu gap). This parses BOTH (json, then
ast.literal_eval) and breaks down label==0 by confidence. Expect Tulu
label==0 & conf==10 ~= the paper's 356,886, which the data.py fix recovers.

In [ ]:
import json, ast
from collections import Counter

def robust_parse(s):
    if isinstance(s, dict):
        return s, "dict"
    for fn, tag in ((json.loads, "json"), (ast.literal_eval, "ast")):
        try:
            return fn(s), tag
        except Exception:
            pass
    return None, "fail"

for src in ["Tulu-3 SFT mixture", "GPT-3 self-generated", "LLMs-from-scratch"]:
    rows = [r for r in ds if r["source"] == src]
    methods, conf_dist, n0, samples = Counter(), Counter(), 0, []
    for r in rows:
        obj, tag = robust_parse(r["label"])
        methods[tag] += 1
        if tag == "fail":
            if len(samples) < 3: samples.append(r["label"])
            continue
        if obj.get("label") == 0:
            n0 += 1
            conf_dist[obj.get("confidence")] += 1
    print(f"\n=== {src}: n={len(rows):,} ===")
    print("  parse methods:", dict(methods))
    print("  label==0:", f"{n0:,}", "| conf==10 among them:", f"{conf_dist.get(10,0):,}")
    print("  confidence dist (label==0):",
          dict(sorted(conf_dist.items(), key=lambda x: -(x[0] if x[0] is not None else -1))))
    if samples: print("  unparseable samples:", [repr(s)[:120] for s in samples])

### 5. Stage source mapping
Confirm the `sources` substrings in `configs/train.yaml` actually match the real
`source` values, and that per-stage counts line up with the paper.

In [ ]:
import yaml
from chrono_instruct.data import stage_examples

cfg = yaml.safe_load(open("configs/train.yaml"))
filtered = [r for r in ds if keep_row(r)]
print(f"filtered rows: {len(filtered):,}\n")
for stage in cfg["stages"]:
    n = sum(1 for _ in stage_examples(filtered, stage["sources"]))
    print(f"{stage['name']:<22} sources={stage['sources']} -> {n:,}")
print("\npaper: scratch 1,097 | self-instruct 67,136 | tulu 356,886")

### 6. Prompt rendering + response-mask correctness
Show a with-input and a no-input rendering, then confirm the loss mask covers
exactly the response (+ EOT) and nothing of the prompt.

In [ ]:
import json as _json
from chrono_instruct.data import format_example, encode_example, ENC

def as_dict(c):
    return _json.loads(c) if isinstance(c, str) else c

def find(has_input):
    for r in filtered:
        c = as_dict(r["conversation"])
        if bool((c.get("input") or "").strip()) == has_input:
            return c

for tag, ex in [("WITH input", find(True)), ("NO input", find(False))]:
    prompt, output = format_example(ex)
    print(f"\n===== {tag} =====\nPROMPT:\n{prompt}\nOUTPUT:\n{output[:200]}")

ids, mask = encode_example(find(True))
resp = ENC.decode([t for t, m in zip(ids, mask) if m])
print(f"\ntokens={len(ids)}  response-share={sum(mask)/len(ids):.3f}")
print("decoded loss (response) tokens:", repr(resp[:200]))  # should == output + <|endoftext|>

### 7. Example length vs block size — the packing/truncation evidence
Quantifies how often examples exceed `BLOCK_SIZE` (=> get split by packing).
Expect Stage 3 (Tulu) to be the worst offender (paper avg length 2513 > 1792).

In [ ]:
import numpy as np
from collections import defaultdict

def stage_of(srcval):
    for stage in cfg["stages"]:
        if any(n.lower() in (srcval or "").lower() for n in stage["sources"]):
            return stage["name"]
    return "other"

lens = defaultdict(list)
for r in filtered:
    ids, _ = encode_example(as_dict(r["conversation"]))
    lens[stage_of(r.get("source"))].append(len(ids))

for st, L in lens.items():
    L = np.array(L)
    print(f"{st:<22} n={len(L):>7,}  mean={L.mean():7.1f}  p50={np.percentile(L,50):6.0f}"
          f"  p95={np.percentile(L,95):6.0f}  max={L.max():6d}  >{BLOCK_SIZE}: {(L>BLOCK_SIZE).mean()*100:5.1f}%")

### 8. Packing sanity
Run `pack_blocks` on a small slice and report blocks produced.

In [ ]:
from chrono_instruct.data import pack_blocks

sample = [as_dict(r["conversation"]) for r in filtered[:2000]]
blocks = pack_blocks(sample, BLOCK_SIZE)
total_tokens = sum(len(encode_example(c)[0]) for c in sample)
print(f"{len(sample)} examples ({total_tokens:,} tokens) -> {len(blocks)} blocks of {BLOCK_SIZE}")
print("first block shapes:", len(blocks[0][0]), len(blocks[0][1]))

### 9. Model load + forward returns (logits, layer_outputs)
Confirm our `model.py` loads a real checkpoint and returns BOTH logits and the
per-layer hidden states (needed for `embed`).

In [ ]:
from chrono_instruct.model import ChronoGPT

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ChronoGPT.from_pretrained(REPO_INSTRUCT).to(device).eval()
ids = torch.tensor(ENC.encode("The capital of France is"), device=device)[None]
with torch.no_grad():
    out = model(ids)
logits, layer_outputs = out
print("returns tuple of", len(out))
print("logits:", tuple(logits.shape), "finite:", bool(torch.isfinite(logits).all()))
print("layer_outputs:", len(layer_outputs), "each", tuple(layer_outputs[0].shape))
print("param dtype:", next(model.parameters()).dtype)  # fp32 expected (correct for AdamW)

### 10. Numerical parity vs the official `ChronoGPT_inference.py`
The strongest correctness check: our vendored model should produce the SAME logits
as the authors' file (our only changes were removing `@inference_mode` and the
KV-cache). Expect a max abs diff ~0.

In [ ]:
import importlib.util

off_path = hf_hub_download(REPO_BASE, "ChronoGPT_inference.py")
spec = importlib.util.spec_from_file_location("official_chrono", off_path)
official = importlib.util.module_from_spec(spec)
spec.loader.exec_module(official)

off_model = official.ChronoGPT.from_pretrained(REPO_BASE).to(device).eval()
our_model = ChronoGPT.from_pretrained(REPO_BASE).to(device).eval()
with torch.no_grad():
    o = off_model(ids)
    o = o[0] if isinstance(o, tuple) else o
    r = our_model(ids)[0]
print("max abs logit diff (our vs official):", (o.float() - r.float()).abs().max().item())
del off_model, our_model
torch.cuda.empty_cache()

### 11. Generation + embedding sanity (our `infer`)

In [ ]:
from chrono_instruct.infer import load, generate, embed

m, dev = load(REPO_INSTRUCT)
print(generate(m, dev, "### Instruction:\nName three primary colors.\n\n### Response:\n", max_new_tokens=40))
v = embed(m, dev, "Inflation is", layer=-1)
print("embedding shape:", tuple(v.shape))

### 12. Prompt format: our Alpaca template vs the official `extract_response`
Generate from the SAME instruct model with both prompt formats and eyeball which
is better. This informs the pending decision in `docs/implementation-notes.md` §1.

In [ ]:
from chrono_instruct.data import PROMPT_NO_INPUT

instr = "Explain what inflation is in one sentence."
ours = PROMPT_NO_INPUT.format(instruction=instr)
sys_prompt = ("You are ChronoGPT, a large language model trained by ManelaLab at WashU.\n"
              "    Below is an instruction that describes a task.\n"
              "    Write a response that appropriately completes the request.")
official = f"\n\n### Instruction:\n{sys_prompt}\n{instr}\n\n### Input:\n### Response:\n"

for tag, p in [("OURS (Alpaca template)", ours), ("OFFICIAL (extract_response)", official)]:
    print(f"\n===== {tag} =====")
    print(generate(m, dev, p, max_new_tokens=60))

### 13. Full fine-tuning fits on the card
One real forward+backward+step on the 1.55B model; report peak VRAM. Confirms full
FT (no PEFT) fits at this batch/block on 80GB.

In [ ]:
from chrono_instruct.train import masked_lm_loss

ft = ChronoGPT.from_pretrained(REPO_BASE).to(device).train()
opt = torch.optim.AdamW(ft.parameters(), lr=1e-5)
batch = torch.randint(0, enc.n_vocab, (1, BLOCK_SIZE), device=device)
labels = batch.clone(); labels[:, : BLOCK_SIZE // 2] = -100
torch.cuda.reset_peak_memory_stats()
with torch.autocast("cuda", dtype=torch.bfloat16):
    logits, _ = ft(batch)
    loss = masked_lm_loss(logits, labels)
loss.backward(); opt.step()
print("loss:", float(loss), "| peak VRAM GB:", round(torch.cuda.max_memory_allocated() / 1e9, 1))
del ft, opt; torch.cuda.empty_cache()

### 14. Smoke test
The CPU end-to-end test (tiny model, asserts the loss decreases).

In [ ]:
import subprocess, sys
# use the kernel's interpreter so pytest is found even if it is not on PATH
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_smoke.py"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)